# Understanding SVM Kernels for Job Classification

This notebook explores how different kernel functions in Support Vector Machines (SVM) affect classification performance.

We use a real-world job dataset where each job is represented by required skills, and the goal is to classify the job into categories.

The focus of this tutorial is to compare:
- Linear Kernel
- Polynomial Kernel
- RBF Kernel

and understand how model complexity impacts performance.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt


## Loading the Dataset

We begin by loading the dataset and inspecting its structure.

In [ ]:
df = pd.read_csv("data/final_data.csv")

print("First 5 rows:\n", df.head())
print("\nShape:", df.shape)
print("\nColumns:\n", df.columns)

## Data Exploration

We check:
- data types
- missing values
- class distribution

In [ ]:
print(df.info())
print("\nMissing Values:\n", df.isnull().sum())
print("\nClass Distribution:\n", df['Class'].value_counts())

## Class Distribution

This shows how balanced or imbalanced the dataset is.

In [ ]:
plt.figure()
df['Class'].value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

## Feature Selection

Only numerical features are used for training the SVM model.

Categorical columns are removed to keep the focus on kernel behaviour and ensure compatibility with the model.

In [ ]:
X = df.select_dtypes(include=['int64', 'float64'])
y = df['Class']

X = X.fillna(0)

print("Features used:\n", X.columns)

## Train-Test Split

The dataset is split into training and testing sets to evaluate model performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Model Training

We train three SVM models using different kernels:
- Linear
- Polynomial
- RBF

In [ ]:
svm_linear = SVC(kernel='linear')
svm_poly = SVC(kernel='poly', degree=3)
svm_rbf = SVC(kernel='rbf')

svm_linear.fit(X_train, y_train)
svm_poly.fit(X_train, y_train)
svm_rbf.fit(X_train, y_train)

## Model Evaluation

We evaluate each model using:
- Accuracy
- Classification report
- Confusion matrix

In [ ]:
def evaluate_model(model, name):
    y_pred = model.predict(X_test)

    print(f"\n{name} Kernel Results:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
evaluate_model(svm_linear, "Linear")
evaluate_model(svm_poly, "Polynomial")
evaluate_model(svm_rbf, "RBF")

## Accuracy Comparison

This plot compares the performance of different kernels.

In [ ]:
accuracies = []

models = {
    "Linear": svm_linear,
    "Polynomial": svm_poly,
    "RBF": svm_rbf
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))

plt.figure()
plt.bar(models.keys(), accuracies)
plt.title("SVM Kernel Comparison")
plt.xlabel("Kernel Type")
plt.ylabel("Accuracy")
plt.show()

## Cross Validation

Cross-validation is used to check whether the model generalises well across different splits of the data.

In [ ]:
def cross_validate_model(model, name):
    scores = cross_val_score(model, X, y, cv=5)

    print(f"\n{name} Cross Validation:")
    print("Scores:", scores)
    print("Mean Accuracy:", scores.mean())

cross_validate_model(SVC(kernel='linear'), "Linear")
cross_validate_model(SVC(kernel='poly', degree=3), "Polynomial")
cross_validate_model(SVC(kernel='rbf'), "RBF")

## Feature Importance

For the linear kernel, feature weights help us understand which skills influence predictions.

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Weight': svm_linear.coef_[0]
})

feature_importance = feature_importance.sort_values(by='Weight', ascending=False)

print(feature_importance.head(10))

## Final Observations

- The Linear SVM achieved perfect accuracy, indicating that the dataset is largely linearly separable.
- The Polynomial kernel performed poorly, likely due to overfitting and sensitivity to class imbalance.
- The RBF kernel performed very well but did not significantly outperform the linear model.

This shows that increasing model complexity does not always improve performance. In this case, a simpler model was sufficient due to the structure of the dataset.